# 03 · Features, split temporal e modelos

**Previsão de atraso de entrega no e-commerce brasileiro (Olist)** · Notebook 3 de 3

Este notebook cobre a etapa de modelagem: as duas features históricas do vendedor (construção hermética anti-vazamento, o ponto técnico central do projeto), o split temporal com corte medido por volumetria, e a comparação de três modelos. **Regra de ouro desta etapa: o conjunto de TESTE não é tocado aqui.** Toda seleção (hiperparâmetros, modelo campeão e, na próxima etapa, threshold) usa um bloco de validação temporal DENTRO do treino; o teste será usado uma única vez, na avaliação final.

O código reutilizável vive em `src/features.py` (construção das features, com os porquês em docstring) e `src/train.py` (split, pipelines e grades de modelos). Aqui o notebook narra e executa.

In [1]:
import json
import subprocess
import warnings

warnings.filterwarnings("ignore", message="X does not have valid feature names")
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.features import build_seller_history_features
from src.train import (
    CATEGORICAL_FEATURES,
    FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    build_model_grid,
    describe_params,
    evaluate,
    temporal_split,
)

master = pd.read_parquet(ROOT / "data" / "processed" / "master_table.parquet")
items = pd.read_csv(ROOT / "data" / "raw" / "olist_order_items_dataset.csv")
print(f"tabela mestra: {len(master):,} pedidos · taxa {master[TARGET].mean():.2%}")
print(f"features na whitelist: {len(FEATURES)} ({len(NUMERIC_FEATURES)} numéricas + {len(CATEGORICAL_FEATURES)} categóricas)")

tabela mestra: 96,470 pedidos · taxa 6.77%
features na whitelist: 21 (18 numéricas + 3 categóricas)


## 1. As features históricas do vendedor (construção hermética)

A EDA mostrou que o passado do vendedor separa personas (gargalos com 16,5% de atraso e postagem de 7,4 dias). Para transformar isso em previsão por pedido, duas features resumem o histórico de cada vendedor **no instante exato de cada compra**: a taxa de atraso passada e o tempo médio de postagem passado.

**Por que o `shift(1)` clássico não basta.** A receita padrão (ordenar por compra e deslocar uma linha) tem um furo: o desfecho do pedido anterior só passa a existir quando ele é **entregue**, ~10 dias depois da compra. Um pedido comprado 2 dias após o anterior usaria, no treino, uma informação que ninguém tinha naquele instante. Em produção, a feature store não teria esse valor, e o modelo veria uma feature mais informativa no treino do que na vida real.

**A construção adotada: cada informação entra no histórico no instante em que nasce.**

- O **desfecho** (atrasou ou não) fica disponível na data de **entrega ao cliente**;
- O **tempo de postagem** fica disponível na data de **entrega à transportadora** (relógio mais cedo: a postagem é observável antes do desfecho).

Para o pedido comprado em $t$, contam apenas eventos com disponibilidade estritamente anterior a $t$. Implementação vetorizada: por vendedor, eventos ordenados por disponibilidade com soma acumulada + busca binária ($O(n \log n)$; detalhes em `src/features.py`).

**Cold start:** vendedor sem entrega anterior recebe a média GLOBAL histórica (mesma regra hermética, todos os vendedores) + flag `seller_no_history`. No comecinho do dataset nem o global existe: fica NaN, tratado pelo imputador de mediana treinado só no treino. **Multi-vendedor:** máximo entre os vendedores do pedido (elo mais fraco), lendo TODOS os vendedores, não só o principal.

O contrato anti-vazamento desta feature é **executável**: 6 testes pytest em `tests/test_features.py` cobrem os casos críticos, incluindo "o desfecho é invisível antes da entrega" e "o relógio da postagem corre antes do relógio do desfecho".

In [2]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", str(ROOT / "tests"), "-q"],
    capture_output=True, text=True, cwd=ROOT,
)
print(result.stdout.strip().splitlines()[-1])
assert result.returncode == 0, "testes da feature crítica falharam"

6 passed in 0.65s


In [3]:
hist = build_seller_history_features(master, items)
master = master.merge(hist, on="order_id", how="left")

print(f"pedidos com flag seller_no_history: {master['seller_no_history'].mean():.2%}")
print(f"NaN residual (início do dataset, sem histórico nem global): {master['seller_late_rate_hist'].isna().sum()}")
print("\nmédia da feature por classe do alvo:")
print(master.groupby(TARGET)[["seller_late_rate_hist", "seller_posting_days_hist"]].mean().round(4).to_string())

pedidos com flag seller_no_history: 5.50%
NaN residual (início do dataset, sem histórico nem global): 266

média da feature por classe do alvo:
         seller_late_rate_hist  seller_posting_days_hist
is_late                                                 
0                       0.0437                    3.1633
1                       0.0493                    3.4958


A separação univariada é modesta (pedidos que atrasaram vêm de vendedores com histórico pior, 4,9% vs 4,4%, e postagem passada mais lenta), o que é esperado: a feature trabalha em conjunto com distância, janela e mês, e sua contribuição real será medida na importância de features da etapa de avaliação.

## 2. Split temporal com corte medido (e um conflito de volumetria resolvido)

Decisões já tomadas: teste truncado em **2018-09-01** (a cobertura de entregas zera em setembro, viés de sobrevivência medido no notebook 01) e aspiração de ~80/20 com corte perto de março/2018, mantendo a greve dos caminhoneiros (maio/2018) dentro do teste.

A volumetria real revela que **as duas aspirações são incompatíveis**: como o volume cresce ao longo do tempo, o ponto que deixaria 80% dos pedidos no treino cai no FIM de maio de 2018, o que colocaria a greve no treino e reduziria o teste ao regime recalibrado pós-greve, destruindo a âncora narrativa do split. Entre a fração e a âncora, fica a âncora: **corte em 2018-03-01**, com split real de ~59/41. Trade-off aceito e documentado: menos treino (modelos um pouco piores), mais teste (métricas mais estáveis), e um teste que contém a saída da crise de fev–mar, a greve de maio e a recalibração de junho, exatamente os regimes que a produção enfrentaria.

In [4]:
TEST_END = "2018-09-01"
CUT = "2018-03-01"
VAL_START = "2018-01-01"

pop = master[master["order_purchase_timestamp"] < pd.Timestamp(TEST_END)].copy()
q80 = pop["order_purchase_timestamp"].quantile(0.8)
print(f"quantil 80% das compras: {q80:%Y-%m-%d} (incompatível com a greve no teste)")

rows = []
for cut in ("2018-03-01", "2018-04-01", "2018-06-01"):
    tr, te = temporal_split(pop, cut, TEST_END)
    rows.append({
        "corte": cut,
        "treino": len(tr), "treino_%": round(100 * len(tr) / len(pop), 1),
        "taxa_treino_%": round(100 * tr[TARGET].mean(), 2),
        "teste": len(te), "taxa_teste_%": round(100 * te[TARGET].mean(), 2),
        "greve_no_teste": cut <= "2018-05-01",
    })
pd.DataFrame(rows).set_index("corte")

quantil 80% das compras: 2018-05-26 (incompatível com a greve no teste)


,treino,treino_%,taxa_treino_%,teste,taxa_teste_%,greve_no_teste
corte,,,,,,
2018-03-01,57317,59.4,6.60,39153,7.02,True
2018-04-01,64320,66.7,7.95,32150,4.42,True
2018-06-01,77867,80.7,7.53,18603,3.61,False


In [5]:
train, test = temporal_split(pop, CUT, TEST_END)
core, val = temporal_split(train, VAL_START, CUT)

print(f"treino: {len(train):,} pedidos até {CUT} (taxa {train[TARGET].mean():.2%})")
print(f"  ├─ núcleo de treino: {len(core):,} até {VAL_START} (taxa {core[TARGET].mean():.2%})")
print(f"  └─ validação temporal: {len(val):,} de jan–fev/2018 (taxa {val[TARGET].mean():.2%})")
print(f"teste (INTOCADO nesta etapa): {len(test):,} de {CUT} a {TEST_END} (taxa {test[TARGET].mean():.2%})")

treino: 57,317 pedidos até 2018-03-01 (taxa 6.60%)
  ├─ núcleo de treino: 43,693 até 2018-01-01 (taxa 5.62%)
  └─ validação temporal: 13,624 de jan–fev/2018 (taxa 9.75%)
teste (INTOCADO nesta etapa): 39,153 de 2018-03-01 a 2018-09-01 (taxa 7.02%)


Nota sobre o bloco de validação: jan–fev/2018 é justamente a **crise logística** vista na EDA (taxa de 9,8% contra 5,6% do núcleo). Hiperparâmetros e threshold serão escolhidos, portanto, sob estresse de mudança de regime, uma escolha conservadora e realista: é exatamente o tipo de deslocamento que o modelo enfrentará no teste e em produção.

## 3. Três modelos, um pipeline, seleção na validação

Cada candidato é um pipeline sklearn único: imputação de mediana **aprendida só no fold de treino** (fecha a decisão da etapa 2 de não imputar na tabela mestra), one-hot para UF/categoria/tipo de pagamento, e o classificador. Desbalanceamento por pesos de classe (`class_weight`/`scale_pos_weight`), nunca reamostragem sintética: preserva a distribuição real e deixa a decisão de custo para o threshold, onde ela pertence ao negócio.

Grade pequena e manual (11 ajustes no total), critério de seleção **AUC-PR na validação** (classe positiva rara torna a área sob precision-recall mais informativa que a ROC).

In [6]:
X_core, y_core = core[FEATURES], core[TARGET]
X_val, y_val = val[FEATURES], val[TARGET]
scale_pos_weight = float((y_core == 0).sum() / (y_core == 1).sum())

results = []
fitted = {}
for family, grid in build_model_grid(scale_pos_weight).items():
    for pipe in grid:
        pipe.fit(X_core, y_core)
        metrics = evaluate(pipe, X_val, y_val)
        results.append({"família": family, "hiperparâmetros": describe_params(pipe), **metrics})
        fitted[(family, describe_params(pipe))] = pipe

results = pd.DataFrame(results)
best = results.loc[results.groupby("família")["auc_pr"].idxmax()]
results.sort_values("auc_pr", ascending=False).round(4).reset_index(drop=True)

,família,hiperparâmetros,auc_pr,auc_roc
0,Regressão Logística,C=0.1,0.2282,0.7165
1,Regressão Logística,C=1.0,0.2227,0.7117
2,Regressão Logística,C=10.0,0.2190,0.7102
3,Random Forest,"depth=8, leaf=20",0.1924,0.6882
4,LightGBM,"n=300, leaves=15",0.1919,0.6784
5,Random Forest,"depth=12, leaf=20",0.1890,0.6839
6,Random Forest,"depth=16, leaf=5",0.1887,0.6788
7,Random Forest,"depth=None, leaf=20",0.1852,0.6766
8,LightGBM,"n=300, leaves=31",0.1849,0.6661
9,LightGBM,"n=600, leaves=31",0.1774,0.6490


In [7]:
best.sort_values("auc_pr", ascending=False).round(4).set_index("família")

,hiperparâmetros,auc_pr,auc_roc
família,,,
Regressão Logística,C=0.1,0.2282,0.7165
Random Forest,"depth=8, leaf=20",0.1924,0.6882
LightGBM,"n=300, leaves=15",0.1919,0.6784


**Leitura honesta do resultado: o baseline vence.** A Regressão Logística (C=0,1, a mais regularizada da grade) supera Random Forest e LightGBM em AUC-PR na validação, e dentro de cada família de árvores as configurações MAIS simples são as melhores (LightGBM prefere 15 folhas a 63; a floresta prefere profundidade 8). Tudo aponta o mesmo mecanismo: entre o núcleo de treino (2016–2017, taxa 5,6%) e a validação (crise de jan–fev/2018, taxa 9,8%) há uma mudança de regime, e modelos de maior capacidade memorizam padrões específicos do regime antigo que não se transferem; o modelo linear, mais rígido, carrega estrutura global (distância, janela, histórico do vendedor) que sobrevive à mudança.

Não é um resultado constrangedor, é um resultado informativo: **complexidade extra não comprou desempenho neste problema com este split honesto**. Com split aleatório (que mistura futuro no treino), as árvores provavelmente venceriam, e essa vitória seria uma ilusão de vazamento temporal. Os três finalistas (o melhor de cada família) seguem para a avaliação única no teste, na próxima etapa.

## 4. Teste de sanidade: o alvo embaralhado

Prova executável do contrato anti-vazamento: treinando o melhor LightGBM com o alvo **permutado aleatoriamente**, nenhuma feature pode conter informação sobre esse alvo falso, e a AUC-ROC na validação deve cair para ~0,5 (moeda ao ar). Se ficasse alta, alguma coluna estaria entregando a resposta por construção.

In [8]:
rng = np.random.default_rng(42)
y_shuffled = pd.Series(rng.permutation(y_core.to_numpy()), index=y_core.index)

best_lgbm_params = best.loc[best["família"] == "LightGBM", "hiperparâmetros"].iloc[0]
sanity_pipe = [
    p for p in build_model_grid(scale_pos_weight)["LightGBM"]
    if describe_params(p) == best_lgbm_params
][0]
sanity_pipe.fit(X_core, y_shuffled)
auc_shuffled = evaluate(sanity_pipe, X_val, y_val)["auc_roc"]
print(f"AUC-ROC com alvo embaralhado: {auc_shuffled:.3f} (esperado ≈ 0,5)")
assert 0.45 < auc_shuffled < 0.55, "possível vazamento estrutural"

AUC-ROC com alvo embaralhado: 0.500 (esperado ≈ 0,5)


## 5. Finalistas reajustados no treino completo e persistidos

Escolhidos os hiperparâmetros na validação, cada finalista é reajustado no **treino inteiro** (núcleo + validação, até o corte) para aproveitar todos os dados disponíveis, e persistido para a etapa de avaliação, onde o teste será tocado uma única vez.

In [9]:
import joblib

MODELS_DIR = ROOT / "data" / "processed" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train, y_train = train[FEATURES], train[TARGET]
for _, row in best.iterrows():
    pipe = fitted[(row["família"], row["hiperparâmetros"])]
    pipe.fit(X_train, y_train)
    slug = row["família"].lower().replace(" ", "_").replace("ã", "a").replace("í", "i")
    joblib.dump(pipe, MODELS_DIR / f"{slug}.joblib")
    print(f"salvo: {slug}.joblib ({row['hiperparâmetros']})")

split_config = {"cut": CUT, "test_end": TEST_END, "val_start": VAL_START}
(MODELS_DIR / "split_config.json").write_text(json.dumps(split_config))
print("split_config.json salvo:", split_config)

salvo: lightgbm.joblib (n=300, leaves=15)


salvo: random_forest.joblib (depth=8, leaf=20)


salvo: regressao_logistica.joblib (C=0.1)
split_config.json salvo: {'cut': '2018-03-01', 'test_end': '2018-09-01', 'val_start': '2018-01-01'}


## 6. Conclusões da etapa e handoff

1. **Features históricas herméticas construídas e testadas:** cada informação entra no instante em que nasce (desfecho na entrega, postagem no despacho); 6 testes pytest verdes cobrem o contrato, incluindo o caso que o `shift(1)` clássico erraria.
2. **Split temporal fixado por volumetria com trade-off explícito:** corte 2018-03-01 (59/41), âncora narrativa preservada (crise, greve e recalibração dentro do teste), teste truncado em 2018-09-01.
3. **Três modelos comparados na validação temporal:** o baseline linear vence sob mudança de regime; complexidade extra não comprou desempenho com split honesto.
4. **Sanidade verificada:** alvo embaralhado derruba a AUC para ~0,5.
5. **Teste ainda intocado.** Próxima etapa: avaliação única no teste com estratificação mensal (crise → greve → recalibração), escolha de threshold na validação, cenário de negócio em contagens e importância de features.